# Think on the Graph, Stage 1: explore a code-only subgraph

Load the knowledge graph, run the concept patterns over one scenario's Java code (nothing else is read), and look at the subgraph that comes out.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from utils import load_kg, load_scenarios, reader_blocks
from preprocessing import mask_java, split_statements
from model import tag_code, subgraph_from_code

kg = load_kg(ROOT / 'data/raw/chapter1_kg.json')
scenarios = {s['id']: s for s in load_scenarios(ROOT / 'data/raw/scenarios_20.json')}
print(kg['version']); print(len(kg['concepts']), 'concepts,', len(kg['edges']), 'edges'); print(sorted(scenarios))

## 1. The code of one scenario, and its preprocessed form

In [ ]:
sid = 'S007'
code = scenarios[sid]['code']
print(code)
print('--- masked (literals and comments blanked, same length) ---')
print(mask_java(code))
print('--- statements ---')
for s in split_statements(mask_java(code)): print(repr(s.strip()))

## 2. Which concept patterns fire

Every hit names the channel (pattern / signal / operator / type / command), the regex or signal name, and the line.

In [ ]:
tags = tag_code(code, reader_blocks(kg))
for cid, hits in sorted(tags.items()):
    generic = (next(c for c in kg['concepts'] if c['id'] == cid).get('reader') or {}).get('generic', False)
    print(f"{cid:28} {'generic' if generic else '':8} " + '; '.join(f"{h['channel']}:{h['name'][:40]} (line {h['line']})" for h in hits[:3]))

## 3. The subgraph

Concept nodes are the tagged concepts minus bare generic ones; then the KG edges between them, their misconception / error nodes, and their level nodes L0..L4.

In [ ]:
sg = subgraph_from_code(code, kg, sid)
print(json.dumps(sg['counts'], indent=1))
print('concept nodes:', [c['id'] + (' (generic)' if c['generic'] else '') for c in sg['concepts']])
print('pieces:', sg['components'])
for e in sg['concept_edges']: print(f"  {e['id']:45} {e['logic'][:90] if e['logic'] else ''}")
print('misconception / error nodes:', [m['id'] for m in sg['misconceptions']])
for w in sg['warnings']: print('note:', w)

## 4. The figure produced by `src/evaluate.py`

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'results/figures' / f'{sid}.png')))

## 5. Try your own snippet

Change the code below; the subgraph comes from the code alone.

In [ ]:
snippet = '''public class Demo {
    public static void main(String[] args) {
        int n = Integer.parseInt(args[0]);
        for (int i = 0; i < n; i++)
            System.out.println(i % 3);
    }
}'''
sg2 = subgraph_from_code(snippet, kg)
print([c['id'] for c in sg2['concepts']])
print([e['id'] for e in sg2['concept_edges']])
print(sg2['counts'])